Databricks notebook source
MAGIC %md
MAGIC
MAGIC ## UMAP for patients with male infertility vs patients without male infertility diagnosis (but can include patients who have had vasectomy)
MAGIC
MAGIC ### This creates UMAPs for patients' phenotypic profiles < 6 months after diagnosis/procedure
MAGIC
MAGIC Note: This notebook includes all diagnoses

COMMAND ----------

In [1]:
import os
os.chdir('/Users/fengxie/Documents/Logistic_Regression_Python_Stanford/Logistic_Regression_Python_MI')
from MI_Functions import *

COMMAND ----------

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy.core.multiarray
import numpy as np
import os
import umap
import re
import scipy
from scipy import stats
from scipy.stats import mstats
from scikit_posthocs import posthoc_dunn
import matplotlib
import re

COMMAND ----------

MAGIC %md
MAGIC ## 'Import' functions

COMMAND ----------

MAGIC %run MI_Functions.py

COMMAND ----------

MAGIC %md
MAGIC ## Read in demographics and diagnoses files for male infertility and vasectomy patients

COMMAND ----------

MAGIC %md
MAGIC ### Demographics

COMMAND ----------

In [3]:
demographics = dict()

In [4]:
demos = ['mi',
         'vas_only']
file_names_demos = ['mi_pts_only_final',
                    'vas_pts_only_final']

In [5]:
for demo, file_name_demo in zip(demos, file_names_demos):
  temp = pd.read_pickle(f"male_infertility_validation/demographics/{file_name_demo}.pkl")
  demographics[demo] = temp

COMMAND ----------

MAGIC %md
MAGIC ### Diagnoses

COMMAND ----------

In [6]:
diagnoses = dict()

In [7]:
diags = ['mi_diag', 
         'vas_only_diag']
file_names_diags = ['mi_phe', 
                    'vas_phe']

In [8]:
for diag, file_name_diag in zip(diags, file_names_diags):
  temp = pd.read_pickle(f"male_infertility_validation/diagnoses/{file_name_diag}.pkl")
  diagnoses[diag] = temp

COMMAND ----------

MAGIC %md
MAGIC #### Filter for phenotypes occurring < 6 months after diagnosis/procedure

COMMAND ----------

In [9]:
diagnoses_before = dict()

In [10]:
# 'mi_diag'
temp = diagnoses['mi_diag'].copy()
temp = temp[temp['phe_time_before'] == 1]
diagnoses_before['mi_diag'] = temp

In [11]:
# 'vas_only_diag'
temp = diagnoses['vas_only_diag'].copy()
temp = temp[temp['phe_time_before'] == 1]
diagnoses_before['vas_only_diag'] = temp

COMMAND ----------

MAGIC %md
MAGIC ## Make pivot table containing all patients' phenotypic profiles

COMMAND ----------

In [12]:
pivot_tables = dict()

In [13]:
for file_name_diag in diagnoses_before:
  pivot_tables[file_name_diag] = make_pivot_tables(diagnoses_before[file_name_diag], file_name_diag, n='phenotype')

%%% Making pivot table for mi_diag %%%
Shape of pivot table: (5547, 1432)


phenotype,Abdominal aortic aneurysm,Abdominal hernia,Abdominal pain,Abnormal arterial blood gases,Abnormal chest sounds,Abnormal coagulation profile,Abnormal electrocardiogram [ECG] [EKG],Abnormal findings examination of lungs,Abnormal findings on exam of gastrointestinal tract/ abdominal area,Abnormal findings on study of brain and/or nervous system,...,Vitamin D deficiency,Vitamin deficiency,Vitiligo,Voice disturbance,Von willebrand's disease,Wegener's granulomatosis,Wheezing,potential health hazards related to communicable diseases\npotential health hazards related to communicable diseases,severe protein-calorie malnutrition,male infertility status
person_id,,,,,,,,,,,,,,,,,,,,,
29939065,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
29939067,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
29939074,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1




Done


%%% Making pivot table for vas_only_diag %%%
Shape of pivot table: (2176, 1264)


phenotype,Abdominal aortic aneurysm,Abdominal hernia,Abdominal pain,Abnormal arterial blood gases,Abnormal coagulation profile,Abnormal electrocardiogram [ECG] [EKG],Abnormal findings examination of lungs,Abnormal findings on exam of gastrointestinal tract/ abdominal area,Abnormal findings on study of brain and/or nervous system,Abnormal function study of cardiovascular system,...,Visual field defects,Vitamin B-complex deficiencies,Vitamin D deficiency,Vitamin deficiency,Vitiligo,Voice disturbance,Wheezing,potential health hazards related to communicable diseases\npotential health hazards related to communicable diseases,severe protein-calorie malnutrition,male infertility status
person_id,,,,,,,,,,,,,,,,,,,,,
29940002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
29940245,0,0,1,0,0,1,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
29940249,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0




Done




COMMAND ----------

MAGIC %md
MAGIC ### Concatenate pivot tables

COMMAND ----------

In [14]:
alldiag_pivot = pd.concat([pivot_tables['mi_diag'], pivot_tables['vas_only_diag']], axis=0)

COMMAND ----------

MAGIC %md
MAGIC #### Drop infertility-related conditions

COMMAND ----------

In [15]:
colstodrop = list()

In [16]:
colstodrop.extend(alldiag_pivot.columns[alldiag_pivot.columns.str.contains('infertility|spermia|reproduction',      
                                                                           flags=re.IGNORECASE)])

In [17]:
# remove male infertility status as a column to drop
colstodrop.remove('male infertility status')

In [18]:
colstodrop = set(colstodrop)
print(colstodrop)

{'Infertility, male', 'Persons encountering health services in circumstances related to reproduction', 'Azoospermia and oligospermia', 'Infertility, female', 'Infertility, female, associated with anovulation'}


COMMAND ----------

In [19]:
alldiag_pivot = alldiag_pivot.drop(colstodrop, axis=1)

COMMAND ----------

MAGIC %md
MAGIC ## Make demographic dfs

COMMAND ----------

In [20]:
demographics['mi'].columns

Index(['person_id', 'year_of_birth', 'estimated_age', 'gender_concept_id',
       'gender', 'race_concept_id', 'race', 'ethnicity_concept_id',
       'ethnicity', 'num_visits_total', 'emr_months_total',
       'first_mi_or_vas_date', 'analysis_cutoff_date', 'mi_or_vas_est_age',
       'num_visits_before', 'num_visits_same', 'num_visits_after',
       'emr_months_before', 'emr_months_after'],
      dtype='object')

COMMAND ----------

In [21]:
demographic_cols = ['person_id', 'year_of_birth', 'estimated_age', 'gender', 'race',
                    'ethnicity', #'location_source_value', 'adi', 'adi_category',
                    'num_visits_before', 'emr_months_before']

COMMAND ----------

In [22]:
# Only keep demographic columns and merge
demographics['mi'] = demographics['mi'][demographic_cols].copy()
demographics['vas_only'] = demographics['vas_only'][demographic_cols].copy()

In [23]:
all_demo_df = pd.concat([demographics['mi'], demographics['vas_only']], axis=0, copy=False)

COMMAND ----------

In [24]:
all_demo_df = all_demo_df.drop_duplicates(). \
                          set_index('person_id').reindex(alldiag_pivot.index)

COMMAND ----------

MAGIC %md
MAGIC ## Check that alldiag_pivot and all_demo dfs have the same number of rows, the same index, and fillna with 0.

COMMAND ----------

MAGIC %md
MAGIC ### fillna with 0

COMMAND ----------

In [25]:
all_demo_df = all_demo_df.fillna(0)
alldiag_pivot = alldiag_pivot.fillna(0)

COMMAND ----------

MAGIC %md
MAGIC ### Check the shape of demo and diag dfs

COMMAND ----------

In [26]:
print(f"Shape of all_demo_df is {all_demo_df.shape}")
print(f"Shape of alldiag_pivot is {alldiag_pivot.shape}")
print(f"Number of rows of the dataframes are the same: {all_demo_df.shape[0] == alldiag_pivot.shape[0]}")
print('\n')

Shape of all_demo_df is (7723, 7)
Shape of alldiag_pivot is (7723, 1493)
Number of rows of the dataframes are the same: True




COMMAND ----------

MAGIC %md
MAGIC ### Check whether indices are the same for demo and diag dfs

COMMAND ----------

In [27]:
demo_index = all_demo_df.index
diag_index = alldiag_pivot.index
print(f"indices are the same for all_demo_df and alldiag_pivot: {demo_index.equals(diag_index)}")
print('\n')

indices are the same for all_demo_df and alldiag_pivot: True




COMMAND ----------

MAGIC %md
MAGIC ## Dimensionality Reduction

COMMAND ----------

In [28]:
alldiag_pivot

phenotype,Abdominal aortic aneurysm,Abdominal hernia,Abdominal pain,Abnormal arterial blood gases,Abnormal chest sounds,Abnormal coagulation profile,Abnormal electrocardiogram [ECG] [EKG],Abnormal findings examination of lungs,Abnormal findings on exam of gastrointestinal tract/ abdominal area,Abnormal findings on study of brain and/or nervous system,...,Pulpitis and necrosis of tooth pulp,Respiratory complications,"Rheumatism, unspecified and fibrositis",Schizophrenia and other psychotic disorders,Subarachnoid hemorrhage (injury),Suicide or self-inflicted injury,Toxic effect of (non-ethyl) alcohol and petroleum and other solvents,Unspecified diffuse connective tissue disease,Ventricular fibrillation and flutter,Villonodular synovitis
person_id,,,,,,,,,,,,,,,,,,,,,
29939065,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29939067,0,0,0,0,0.0,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29939074,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29939092,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29939114,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96914433,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
96916587,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97073873,0,0,0,0,0.0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


COMMAND ----------

In [29]:
X = dimensionality_reduction(diag=alldiag_pivot, file_name='mi_vas_only_before')

Peforming dimensionality reduction for mi_vas_only_before...
UMAP(angular_rp_forest=True, metric='cosine', random_state=42, verbose=True)
Fri Sep 15 02:25:20 2023 Construct fuzzy simplicial set
Fri Sep 15 02:25:20 2023 Finding Nearest Neighbors
Fri Sep 15 02:25:20 2023 Building RP forest with 9 trees
Fri Sep 15 02:25:21 2023 NN descent for 13 iterations
	 1  /  13
	 2  /  13
	 3  /  13
	 4  /  13
	 5  /  13
	 6  /  13
	 7  /  13
	 8  /  13
	Stopping threshold met -- exiting after 8 iterations
Fri Sep 15 02:25:26 2023 Finished Nearest Neighbor Search
Fri Sep 15 02:25:27 2023 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

Fri Sep 15 02:25:39 2023 Finished embedding


,index,0,1
7718,7718,-14.374228,6.359439
7719,7719,-18.942024,-0.228846
7720,7720,-20.422915,8.717858
7721,7721,-17.689520,5.364688
7722,7722,-18.285362,8.491445


Saving X as pandas DataFrame...
Saved


COMMAND ----------

MAGIC %md
MAGIC ### Save 'y', which will preserve each patient's male infertility status and other demographic features (it is not preserved after performing dimensionality reduction)

COMMAND ----------

In [30]:
temp = alldiag_pivot.copy()
y = temp['male infertility status'].replace({1 : 'male infertility patient', 0 : 'control (vasectomy patient)'})
y = y.to_frame()
y = y.reset_index()
y = y.reset_index()
y = y.merge(all_demo_df, on='person_id')
y.to_pickle("male_infertility_validation/tables/umap/y_all_before.pkl")

COMMAND ----------

In [31]:
y['male infertility status'].value_counts()

male infertility patient       5547
control (vasectomy patient)    2176
Name: male infertility status, dtype: int64

COMMAND ----------

MAGIC %md
MAGIC ## Save demographic dataframes

COMMAND ----------

In [32]:
temp = all_demo_df.copy()
temp = temp.reset_index()
temp.to_pickle("male_infertility_validation/tables/umap/demo_all_before.pkl")

In [33]:
print(f"Shape of alldiag_pivot is {alldiag_pivot.shape}")

Shape of alldiag_pivot is (7723, 1493)


COMMAND ----------